In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
df = pd.read_parquet('./data/train/1_회원정보_train.parquet')

In [3]:
df['Segment'].value_counts()

Segment
E    1922052
D     349242
C     127590
A        972
B        144
Name: count, dtype: int64

In [4]:
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

### D vs Other 추출하여 `ID`, `기준년월`로 병합

In [3]:
def get_segment_preprocessor(seg_path='./data/train/1_회원정보_train.parquet'):
    """
    Segment 정보 파일에서 ID, 기준년월, Segment 로드 및 전처리 함수 반환

    Returns:
        segment_df (DataFrame): Segment 포함된 기준 정보 (ID + 기준년월 포함)
        preprocess_fn (function): 문자열형 컬럼 숫자형 변환용 전처리 함수
    """
    # 1. Segment 정보 불러오기 
    segment_df = pd.read_parquet(seg_path)[['ID', '기준년월', 'Segment']]
    segment_df['ID'] = segment_df['ID'].astype(str)
    segment_df['기준년월'] = segment_df['기준년월'].astype(str)

    # target 컬럼 추가 (D vs Others)
    segment_df = segment_df[segment_df['Segment'] != 'E']
    segment_df['target'] = (segment_df['Segment'] == 'D').astype(int)


    # 2. 전처리 함수 정의
    def preprocess(df):
        df_cleaned = df.copy()
        obj_cols = df_cleaned.select_dtypes(include='object').columns
        for col in obj_cols:
            sample_values = df_cleaned[col].dropna().astype(str).unique()
            if all(isinstance(v, str) and re.fullmatch(r'\d+대', v) for v in sample_values):
                df_cleaned[col] = pd.to_numeric(df_cleaned[col].str.replace("대", "", regex=False), errors='coerce')
            elif all(isinstance(v, str) and re.fullmatch(r'\d+개', v) for v in sample_values):
                df_cleaned[col] = pd.to_numeric(df_cleaned[col].str.replace("개", "", regex=False), errors='coerce')
            elif all(isinstance(v, str) and re.fullmatch(r'\d+회 이상', v) for v in sample_values):
                df_cleaned[col] = pd.to_numeric(df_cleaned[col].str.extract(r'(\d+)')[0], errors='coerce')
            elif all(isinstance(v, str) and re.fullmatch(r'\d+일 이상', v) for v in sample_values):
                df_cleaned[col] = pd.to_numeric(df_cleaned[col].str.extract(r'(\d+)')[0], errors='coerce')
            elif all(isinstance(v, str) and re.fullmatch(r'\d{2}\.\d+만원\+', v) for v in sample_values):
                df_cleaned[col] = pd.to_numeric(df_cleaned[col].str.extract(r'(\d{2})')[0], errors='coerce')
            elif all(isinstance(v, str) and re.fullmatch(r'[A-Z]', v) for v in sample_values):
                df_cleaned[col] = df_cleaned[col].astype('category').cat.codes
            else:
                df_cleaned[col] = df_cleaned[col].astype('category').cat.codes

        return df_cleaned.fillna(0)

    return segment_df, preprocess

In [ ]:
import os

def run_correlation_isD(paths, segment_df, preprocess):
    save_dir = './corr_output/D_Other/'
    os.makedirs(save_dir, exist_ok=True)

    for path in paths:
        file_name = os.path.basename(path)
        try:
            df = pd.read_parquet(path)
            df['ID'] = df['ID'].astype(str)
            df['기준년월'] = df['기준년월'].astype(str)

            merged = pd.merge(df, segment_df[['ID', '기준년월', 'target']], on=['ID', '기준년월'], how='inner')
            merged = preprocess(merged)

            drop_cols = ['ID', '기준년월']
            if 'Segment' in merged.columns:
                drop_cols.append('Segment')

            corr_df = merged.drop(columns=drop_cols).corr()

            if 'target' not in corr_df.columns:
                print(f"[경고] target 컬럼 없음: {file_name}")
                continue

            corr_series = corr_df['target'].drop('target')
            corr_df = corr_series.to_frame(name='correlation')
            corr_df['abs_correlation'] = corr_df['correlation'].abs()

            # abs(corr) ≥ 0.1 필터링
            filtered = corr_df[corr_df['abs_correlation'] >= 0.1]
            if filtered.empty:
                print(f"[스킵] 의미 있는 상관 피처 없음: {file_name}")
                continue

            # 저장
            output_path = os.path.join(save_dir, f'corr_{file_name.replace(".parquet", ".csv")}')
            filtered.sort_values(by='abs_correlation', ascending=False).to_csv(output_path)

            print(f"[성공] 저장 완료: {output_path}")

        except Exception as e:
            print(f"[에러] {file_name}: {e}")

In [ ]:
# 1. 경로 리스트
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

segment_df, preprocess = get_segment_preprocessor(seg_path=paths[0])  # 1번 파일에는 Segment가 포함되어 있음

# D vs Others 상관계수 분석 실행
run_correlation_isD(paths, segment_df, preprocess)

[성공] 저장 완료: ./corr_output/C_Other/corr_1_회원정보_train.csv
[성공] 저장 완료: ./corr_output/C_Other/corr_2_신용정보_train.csv
[성공] 저장 완료: ./corr_output/C_Other/corr_3_승인매출정보_train.csv
[성공] 저장 완료: ./corr_output/C_Other/corr_4_청구입금정보_train.csv
[성공] 저장 완료: ./corr_output/C_Other/corr_5_잔액정보_train.csv
[스킵] 의미 있는 상관 피처 없음: 6_채널정보_train.parquet
[스킵] 의미 있는 상관 피처 없음: 7_마케팅정보_train.parquet
[성공] 저장 완료: ./corr_output/C_Other/corr_8_성과정보_train.csv


In [8]:
import glob

# 상관계수 결과 파일들 불러오기
corr_files = sorted(glob.glob('./corr_output/D_Other/*train.csv'))

# 파일별로 읽고 통합
df_list = []
for file in corr_files:
    source_name = os.path.basename(file).replace('_corr_is_D.csv', '')  # 파일명만 추출
    df = pd.read_csv(file, index_col=0)
    df = df.reset_index().rename(columns={'index': 'feature', df.columns[0]: 'correlation'})
    df['source'] = source_name
    df_list.append(df)

# 전체 합치기
combined_corr = pd.concat(df_list, ignore_index=True)

# 저장
combined_corr.to_csv('./corr_output/D_Other/전체_is_D_상관계수_통합.csv', index=False, encoding='utf-8-sig')

print("완료: './corr_output/D_Other/전체_is_D_상관계수_통합.csv'")


완료: './corr_output/D_Other/전체_is_D_상관계수_통합.csv'


In [4]:
df = pd.read_csv('./corr_output/전체_is_C_상관계수_통합.csv')

In [5]:
df

,feature,correlation,source
0,기준년월,0.000296,1_회원정보_balanced
1,남녀구분코드,-0.097586,1_회원정보_balanced
2,회원여부_이용가능,0.129855,1_회원정보_balanced
3,회원여부_이용가능_CA,0.123125,1_회원정보_balanced
4,회원여부_이용가능_카드론,-0.016235,1_회원정보_balanced
...,...,...,...
810,변동률_잔액_B1M,0.005137,8_성과정보_balanced
811,변동률_잔액_일시불_B1M,0.010827,8_성과정보_balanced
812,변동률_잔액_CA_B1M,-0.017839,8_성과정보_balanced
813,혜택수혜율_R3M,-0.141749,8_성과정보_balanced


In [39]:
corr_df = df[df['correlation'].abs() >= 0.3]
corr_df

,feature,correlation,source
6,소지카드수_유효_신용,0.403866,1_회원정보_balanced
7,소지카드수_이용가능_신용,0.400320,1_회원정보_balanced
8,입회일자_신용,-0.380032,1_회원정보_balanced
9,입회경과개월수_신용,0.379919,1_회원정보_balanced
22,유효카드수_신용체크,0.393875,1_회원정보_balanced
...,...,...,...
600,평잔_일시불_해외_6M,0.318694,5_잔액정보_balanced
800,잔액_신판ca평균한도소진율_r6m,0.311938,8_성과정보_balanced
801,잔액_신판ca최대한도소진율_r6m,0.339267,8_성과정보_balanced
802,잔액_신판ca평균한도소진율_r3m,0.317474,8_성과정보_balanced


In [41]:
corr_df = df[df['correlation'].abs() >= 0.4]
corr_df

,feature,correlation,source
6,소지카드수_유효_신용,0.403866,1_회원정보_balanced
7,소지카드수_이용가능_신용,0.400320,1_회원정보_balanced
27,이용가능카드수_신용체크,0.406886,1_회원정보_balanced
28,이용가능카드수_신용,0.403892,1_회원정보_balanced
32,이용카드수_신용체크,0.448962,1_회원정보_balanced
...,...,...,...
528,잔액_일시불_B0M,0.407011,5_잔액정보_balanced
534,월중평잔_일시불_B0M,0.432995,5_잔액정보_balanced
583,월중평잔_일시불,0.432407,5_잔액정보_balanced
589,평잔_일시불_3M,0.426620,5_잔액정보_balanced


In [6]:
corr_df = df[df['correlation'].abs() >= 0.5]
corr_df

,feature,correlation,source
37,이용금액_R3M_신용체크,0.612636,1_회원정보_balanced
38,이용금액_R3M_신용,0.578470,1_회원정보_balanced
42,_1순위카드이용금액,0.580535,1_회원정보_balanced
118,이용금액_일시불_B0M,0.582582,3_승인매출정보_balanced
136,이용건수_신용_R12M,0.514756,3_승인매출정보_balanced
137,이용건수_신판_R12M,0.508623,3_승인매출정보_balanced
138,이용건수_일시불_R12M,0.503655,3_승인매출정보_balanced
146,이용금액_일시불_R12M,0.538985,3_승인매출정보_balanced
182,이용금액_일시불_R6M,0.592740,3_승인매출정보_balanced
210,이용금액_일시불_R3M,0.583985,3_승인매출정보_balanced
